# Working with Argovis JSON

For most applications in Python, the [Argovis helpers](https://pypi.org/project/argovisHelpers/) package provides tooling to load profile objects as a specialized Profiles class, and gridded data as xarrays, as illustrated in the introductory notebooks. However, Argovis's API provides data directly as JSON, and for some specialized applications or when working in other languages, you may need to consume this format directly, which we illustrate below by walking through some typical queries and looking at the JSON responses.

## Setup: Register an API key

In order to allocate Argovis's limited computing resources fairly, users are encouraged to register and request a free API key. This works like a password that identifies your requests to Argovis. To do so:

 - Visit [https://argovis-keygen.colorado.edu/](https://argovis-keygen.colorado.edu/)
 - Fill out the form under _New Account Registration_
 - An API key will be emailed to you shortly.
 
Treat this API key like a password - don't share it or leave it anywhere public. If you ever forget it or accidentally reveal it to a third party, see the same website above to change or deactivate your token.

Put your API key in the quotes in the variable below before moving on:

In [68]:
API_ROOT='https://argovis-api.colorado.edu/'
API_KEY=''

# Argovis data structures

Argovis standard data structures divide measurements into _data_ and _metadata_ documents. Typically, a data document corresponds to measurements or gridded data associated with a discreet temporospatial column - a time, latitude and longitude. A single such document may contain measurements at multiple depths or altitudes, provided they share the same latitude, longitude, and time.

Each of these data documents will refer to at least one corresponding metadata document that captures additional information about the measurement. Argovis divides information between data and metadata documents in order to minimize redundancy in the data you download: many data documents will point to the same metadata document, allowing you to only download that metadata once. Typically, these metadata groupings will refer to some meaningful characteristic of the data; Argo metadata documents correspond to physical floats, while CCHDO metadata documents correspond to cruises, for example.

For more detail and specifications on the data and metadata documents for each collection, see [https://argovis.colorado.edu/docs/documentation/_build/html/database/schema.html](https://argovis.colorado.edu/docs/documentation/_build/html/database/schema.html).

# The standard data routes

## What datasets does Argovis index?

Argovis supports several different data sets with the API and data structures described here. They and their corresponding routes are:

 - Argo profiling float data, `/argo`
 - CCHDO ship-based profile data, `/cchdo`
 - tropical cyclone data from HURDAT and JTWC, `/tc`
 - Global Drifter Program data, `/drifters`
 - Easy Ocean, `/easyocean`
 - several gridded products:
   - Roemmich-Gilson total temperature and salinity, `/grids/rg09`
   - LocalGP obervation-based grids, `/grids/localGPintegrals`
   - GLODAP, `/grids/glodap`
 - Argone Argo float position forecast model data, `/argone`
 - Argo trajectory data, `/argotrajectories`
 - several satellite-based timeseries:
   - NOAA sea surface temperature, `/timeseries/noaasst`
   - Copernicus sea surface height, `/timeseries/copernicussla`
   - CCMP wind vector product, `/timeseries/ccmpwind`
   
The examples that follow apply equally to all these routes; they all support similar query options and follow similar behavior patterns.

## Building queries and downloading data

Argovis provides [API documentation](https://argovis.github.io/documentation/api/qsp.html) for all of its data services. These data services are split into several categories, which have three typical routes:

 - The main _data route_, like `/argo`, or `/cchdo`. These routes provide the data documents for the dataset named in the route.
 - The _metadata route_, like `/argo/meta`. These routes provide the metadata documents referred to by data documents.
 - The _vocabulary route_, like `/argo/vocabulary`. These routes provide lists of possible options for search parameters used in the corresponding data and metadata routes.

If you're familiar with REST APIs, this is enough information for you to construct a query string and issue a request in any programming environment that can facilitate an HTTP GET request. If you're working in Python, we provide a helper library, `argovisHelpers`, to manage these requests for you. Let's try it out by making our first request for Argo data, for profiles found within 100 km of a point in the South Atlantic in May 2011 (users of Python's `requests` module will notice a familiar pattern, providing the query string parameters listed in the Swagger docs and associated values as a dictionary):

In [69]:
from argovisHelpers import helpers as avh

In [70]:
argoSearch = {
    'startDate': '2011-05-01T00:00:00Z',
    'endDate': '2011-06-01T00:00:00Z',
    'center': '-22.5,0',
    'radius': 100
}

argoProfiles = avh.query('argo', options=argoSearch, apikey=API_KEY, apiroot=API_ROOT, verbose=True)

https://argovis-api.colorado.edu/argo?startDate=2011-05-01T00:00:00Z&endDate=2011-06-01T00:00:00Z&center=-22.5,0&radius=100


Let's have a look at what we get from the first profile returned:

In [71]:
argoProfiles[0]

{'_id': '1901094_109',
 'geolocation': {'type': 'Point', 'coordinates': [-21.851, 0.602]},
 'basin': 1,
 'timestamp': '2011-05-31T04:48:00.000Z',
 'date_updated_argovis': '2025-01-31T07:45:11.114Z',
 'source': [{'source': ['argo_core'],
   'url': 'ftp://ftp.ifremer.fr/ifremer/argo/dac/coriolis/1901094/profiles/D1901094_109.nc',
   'date_updated': '2025-01-30T15:11:24.000Z'}],
 'cycle_number': 109,
 'geolocation_argoqc': 1,
 'profile_direction': 'A',
 'timestamp_argoqc': 1,
 'vertical_sampling_scheme': 'Primary sampling: averaged [10 sec sampling, 25 dbar average from 2000 dbar to 200 dbar; 10 sec sampling, 10 dbar average from 200 dbar to 10.0 dbar]',
 'data_info': [['pressure',
   'pressure_argoqc',
   'salinity',
   'salinity_argoqc',
   'temperature',
   'temperature_argoqc'],
  ['units', 'data_keys_mode'],
  [['decibar', 'D'],
   [None, None],
   ['psu', 'D'],
   [None, None],
   ['degree_Celsius', 'D'],
   [None, None]]],
 'metadata': ['1901094_m0']}

This is a data document for Argo, matching the specification [in the docs](https://argovis.github.io/documentation/database/schema.html#argo-schema-extension). It contains the `timestamp` and `geolocation` properties that place this profile geospatially, and other parameters that typically change from point to point.

All data documents bear a `metadata` key, which is a pointer to the appropriate metadata document to find out more about this measurement. Let's fetch that document for this first profile by querying the `argo/meta` route for a doument with an `id` that matches this `metadata` pointer:

In [72]:
metaOptions = {
    'id': argoProfiles[0]['metadata'][0]
}

argoMeta = avh.query('argo/meta', options=metaOptions, apikey=API_KEY, apiroot=API_ROOT, verbose=True)
argoMeta

https://argovis-api.colorado.edu/argo/meta?id=1901094_m0


[{'_id': '1901094_m0',
  'data_type': 'oceanicProfile',
  'data_center': 'IF',
  'instrument': 'profiling_float',
  'pi_name': ['Christine Coatanoan'],
  'platform': '1901094',
  'platform_type': 'PROVOR',
  'fleetmonitoring': 'https://fleetmonitoring.euro-argo.eu/float/1901094',
  'oceanops': 'https://www.ocean-ops.org/board/wa/Platform?ref=1901094',
  'positioning_system': 'ARGOS',
  'wmo_inst_type': '841'}]

In addition to temporospatial searches, data and metadata routes typically support _category searches_, which are searches for documents that belong to certain categories. Which categories are available to search by changes logically from dataset to dataset; Argo floats can be searched by platform number, for example, while tropical cyclones can be searched by storm name. See the docs for the full set of possibilities for each category; let's now use argo's platform category search to get all profiles collected by the same platform as the first profile above:

In [73]:
platformSearch = {
    'platform': argoMeta[0]['platform']
}

platformProfiles = avh.query('argo', options=platformSearch, apikey=API_KEY, apiroot=API_ROOT, verbose=True)
print(len(platformProfiles))

https://argovis-api.colorado.edu/argo?platform=1901094
176


For all category searches, we may wish to know the full list of all possible values a category can take on; for this, there are the _vocabulary_ routes. All vocabulary routes support a parameter `enum`, to list what other categorical parameters are available to filter this dataset by:

In [74]:
vocab_enum = {
    'parameter': 'enum'
}

avh.query('argo/vocabulary', options=vocab_enum, apikey=API_KEY, apiroot=API_ROOT, verbose=True)

https://argovis-api.colorado.edu/argo/vocabulary?parameter=enum


['platform', 'source', 'data', 'metadata', 'platform_type', 'position_qc']

Evidently we can filter Argo data by platform, for example. Let's see what platforms are available:

In [75]:
platformVocabSearch = {
    'parameter': 'platform'
}

platforms = avh.query('argo/vocabulary', options=platformVocabSearch, apikey=API_KEY, apiroot=API_ROOT, verbose=True)
print(platforms[0:10])

https://argovis-api.colorado.edu/argo/vocabulary?parameter=platform
['13857', '13858', '13859', '15819', '15820', '15851', '15852', '15853', '15854', '15855']


Here we just print out the first 10 platform IDs found, but all 20 thousand or so are present.

## Using the `data` query option

The astute reader may have noticed something about the data document shown above: there's no actual measurements included in it! By default, only the non-measurement data is returned, in order to minimize bandwidth consumed; in order to get back actual measurements and their QC flags, we must query and filter including the `data` parameter, the behavior of which we'll see in this section.

### Basic data request

Let's start by asking for one particular profile by ID, and ask for some temperature data to go with:

In [76]:
dataQuery = {
    'id': '4901283_003',
    'data': 'temperature'
}

profile = avh.query('argo', options=dataQuery, apikey=API_KEY, apiroot=API_ROOT, verbose=True)
print(profile[0]['data'])

https://argovis-api.colorado.edu/argo?id=4901283_003&data=temperature
[[28.669001, 28.667999, 28.722, 28.816, 28.823, 28.826, 28.830999, 28.783001, 28.775999, 28.740999, 28.694, 28.551001, 28.497, 28.489, 28.414, 28.191999, 28.087999, 28.044001, 27.836, 27.715, 27.655001, 27.41, 27.125999, 26.805, 26.500999, 26.311001, 26.075001, 25.448, 25.120001, 24.632999, 23.709, 22.400999, 21.893, 21.523001, 21.122, 20.861, 20.657, 20.104, 19.707001, 19.681, 19.573999, 19.396999, 19.181, 18.747, 18.438999, 18.240999, 18.045, 17.882, 17.783001, 17.664, 17.563, 17.474001, 17.370001, 17.250999, 17.006001, 16.861, 16.638, 16.465, 16.337999, 16.142, 15.902, 15.721, 15.585, 15.415, 15.28, 15.214, 15.158, 15.084, 15.039, 14.989, 14.91, 14.792, 14.726, 14.698, 14.681, 14.654, 14.618, 14.589, 14.546, 14.459, 14.389, 14.329, 14.219, 14.137, 14.074, 14.039, 14, 13.966, 13.928, 13.851, 13.83, 13.826, 13.823, 13.819, 13.812, 13.808, 13.809, 13.81, 13.808, 13.769, 13.753, 13.736, 13.721, 13.694, 13.657, 13.625,

The `data` key contains a list of lists of measurements. To interpret them, we use the `data_info` key:

In [77]:
print(profile[0]['data_info'])

[['temperature', 'pressure'], ['units', 'data_keys_mode'], [['degree_Celsius', 'D'], ['decibar', 'D']]]


`data_info` is always a list that contains exactly three items:

 - `data_info[0]` is the list of measurements returned in our `data` object, in the same order as `data`. So in the example above, `data[0]` are pressure measurements, while `data[1]` are temperature measurements. Note we got back pressures even though we only asked for temperatures; pressures are always provided where available as they are needed to meaningfully interpret all other data variables.
 - `data_info[1]` is a list of per-measurement variables. In the example above, pressure and temperature both have a `units` and a `data_keys_mode` associated with them.
 - `data_info[2]` is a rank 2 matrix with rows labeled by `data_info[0]` and columns by `data_info[1]`. So for the example above, this matrix indicates pressure has units 'decibar', and temperature has `data_keys_mode` 'D'.
 
With this information, we now understand how to interpret the `data` key above: the first list is a list of pressures measured in decibar, and the second list are corresponding temperature measurements measured in degrees C. Note that the ith elements in the data lists all correspond to the same level - in other words, `data[0][i],  data[1][i], data[2][1], ....` are all measurements corresponding to the ith level of this object.
 
> **Data and metadata precedence**: sometimes, you might see a given key on *both* a data document and its corresponding metadata document; when this happens, the value on the data document always takes precedence. `data_info` is a common example of this, which we'll see again below.

### Getting absolutely everything

What we've seen above allows us to be very targeted in the data we download; rather than being forced to spend time and bandwidth downloading data we aren't interested in, we can focus on just what we need. On the other hand, somtimes we really do want everything, and for that there's `data=all`:

In [78]:
dataQuery = {
    'id': '4901283_003',
    'data': 'all'
}

profile = avh.query('argo', options=dataQuery, apikey=API_KEY, apiroot=API_ROOT, verbose=True)
avh.data_inflate(profile[0])[0:10]

https://argovis-api.colorado.edu/argo?id=4901283_003&data=all


[{'doxy': None,
  'doxy_argoqc': 4,
  'pressure': 2,
  'pressure_argoqc': 1,
  'salinity': 35.574966,
  'salinity_argoqc': 1,
  'salinity_sfile': 35.574966,
  'salinity_sfile_argoqc': 1,
  'temperature': 28.669001,
  'temperature_argoqc': 1,
  'temperature_sfile': 28.669001,
  'temperature_sfile_argoqc': 1},
 {'doxy': None,
  'doxy_argoqc': 4,
  'pressure': 4,
  'pressure_argoqc': 1,
  'salinity': 35.573761,
  'salinity_argoqc': 1,
  'salinity_sfile': 35.573761,
  'salinity_sfile_argoqc': 1,
  'temperature': 28.667999,
  'temperature_argoqc': 1,
  'temperature_sfile': 28.667999,
  'temperature_sfile_argoqc': 1},
 {'doxy': None,
  'doxy_argoqc': 4,
  'pressure': 6,
  'pressure_argoqc': 1,
  'salinity': 35.626602,
  'salinity_argoqc': 1,
  'salinity_sfile': 35.626602,
  'salinity_sfile_argoqc': 1,
  'temperature': 28.722,
  'temperature_argoqc': 1,
  'temperature_sfile': 28.722,
  'temperature_sfile_argoqc': 1},
 {'doxy': None,
  'doxy_argoqc': 4,
  'pressure': 8,
  'pressure_argoqc': 1,

> **Downloading only what you need:**
Some objects, like Argo BGC probes, measure many values. Your downoads will often be dramatically faster if you specify your variables of interest, rather than using `data=all` unnecessarily. Recall that the `data` parameter can also accept a comma-separated list of variable names, if there are a few that you'd like.

### Filtering behavior of data requests

Note that adding a specific data filter is a _firm requirement_ that all returned profiles have some meaningful data for _all_ variables listed. Try demanding chlorophyl-a in addition to temperature for our current profile of interest:

In [79]:
dataQuery = {
    'id': '4901283_003',
    'data': 'temperature,chla'
}

profile = avh.query('argo', options=dataQuery, apikey=API_KEY, apiroot=API_ROOT, verbose=True)
print(profile)

https://argovis-api.colorado.edu/argo?id=4901283_003&data=temperature,chla
[]


We get nothing in our array of profiles; even though we asked for profile id '4901283_003' and we know it exists, `data=temperature,chla` filters our query down to _only_ profiles that have both temperature and chla reported; since the profile requested doesn't have any chla measurements, it is dropped from the returns in this case. This is useful if you only want to download profiles that definitely have data of interest; for example, try the same thing on our regional search from above:

In [80]:
argoSearch = {
    'startDate': '2011-05-01T00:00:00Z',
    'endDate': '2011-06-01T00:00:00Z',
    'center': '-22.5,0',
    'radius': 100,
    'data': 'temperature,chla'
}

argoProfiles = avh.query('argo', options=argoSearch, apikey=API_KEY, apiroot=API_ROOT, verbose=True)
print(len(argoProfiles))

https://argovis-api.colorado.edu/argo?startDate=2011-05-01T00:00:00Z&endDate=2011-06-01T00:00:00Z&center=-22.5,0&radius=100&data=temperature,chla
0


Evidently Argo made no chlorophyl-a measurements in May 2011 within 100 km of our point of interest - a fact which we found using the data api without having to download or reduce any data at all. One final point on data filtering in this manner: it's not enough for a profile to nominally have a variable defined for it; it must have at least one non-null value reported for that variable somewhere in the search results. For example, when we did `data=all` for our profile of interest above, we saw dissolved oxygen, `doxy`, was defined for it. But:

In [81]:
dataQuery = {
    'id': '4901283_003',
    'data': 'doxy'
}

profile = avh.query('argo', options=dataQuery, apikey=API_KEY, apiroot=API_ROOT, verbose=True)
print(profile)

https://argovis-api.colorado.edu/argo?id=4901283_003&data=doxy
[]


Again our search is filtered down to nothing, since every level in that profile reported `None` for `doxy`.

### Search negation

Let's find some profiles that do actually have dissolved oxygen in them, this time with a slightly different geography search: let's look for everything in August 2017 within a polygon region, defined as a list of `[longitude, latitude]` points: 

In [82]:
dataQuery = {
    'startDate': '2017-08-01T00:00:00Z',
    'endDate': '2017-09-01T00:00:00Z',
    'polygon': [[-150,-30],[-155,-30],[-155,-35],[-150,-35],[-150,-30]],
    'data': 'doxy'
}

profiles = avh.query('argo', options=dataQuery, apikey=API_KEY, apiroot=API_ROOT, verbose=True)

https://argovis-api.colorado.edu/argo?startDate=2017-08-01T00:00:00Z&endDate=2017-09-01T00:00:00Z&polygon=[[-150,+-30],+[-155,+-30],+[-155,+-35],+[-150,+-35],+[-150,+-30]]&data=doxy


We find one profile with meaningful dissolved oxygen data in the region of interest.

The `data` key also accepts _tilde negation_, meaning 'filter for profiles that _don't_ contain this data', for example:

In [83]:
dataQuery = {
    'startDate': '2017-08-01T00:00:00Z',
    'endDate': '2017-09-01T00:00:00Z',
    'polygon': [[-150,-30],[-155,-30],[-155,-35],[-150,-35],[-150,-30]],
    'data': 'temperature,~doxy'
}

profiles = avh.query('argo', options=dataQuery, apikey=API_KEY, apiroot=API_ROOT, verbose=True)
print(len(profiles))

https://argovis-api.colorado.edu/argo?startDate=2017-08-01T00:00:00Z&endDate=2017-09-01T00:00:00Z&polygon=[[-150,+-30],+[-155,+-30],+[-155,+-35],+[-150,+-35],+[-150,+-30]]&data=temperature,~doxy
19


We get a collection of profiles that appear in the region of interest, and have temperature but _not_ dissolved oxygen. In this way, we can split up our downloads into groups of related and interesting profiles without re-downloading the same profiles over and over.

### QC filtering

In addition to querying and filtering by what data is available, we can also make demands on the quality of that data by performing QC filtering for datasets that support it. Let's start by looking at some particulate backscattering data:

In [84]:
bbpQuery = {
    'id': '2902857_001',
    'data': 'bbp700,bbp700_argoqc'
}

bbp = avh.query('argo', options=bbpQuery, apikey=API_KEY, apiroot=API_ROOT, verbose=True)
bbpindex = bbp[0]['data_info'][0].index('bbp700')
bbpQCindex = bbp[0]['data_info'][0].index('bbp700_argoqc')
print(bbp[0]['data'][bbpindex][0:10])
print(bbp[0]['data'][bbpQCindex][0:10])

https://argovis-api.colorado.edu/argo?id=2902857_001&data=bbp700,bbp700_argoqc
[0.004157, 0.005183, 0.004465, 0.005632, 0.005568, 0.00426, 0.004298, 0.004439, 0.004568, 0.004593]
[1, 1, 4, 1, 1, 4, 1, 1, 1, 1]


We request both the measurement and its corresponding QC flags, for reference. Recall that for Argo:

 - QC 1 == definitely good data
 - QC 2 == probably good data
 - QC 3 == probably bad data
 - QC 4 == definitely bad data
 
If we didn't look at the QC flags for our particulate backscatter data, we could easily have missed that some of the measurements shown above (and many more in the profile not printed) have been marked as bad data by the upstream data distributor, and therefore might not be appropriate for your purposes. We can suppress measurements based on a list of allowed QC values by modifying what we pass to the `data` query parameter:

In [85]:
bbpQCfilteredQuery = {
    'id': '2902857_001',
    'data': 'bbp700,1,bbp700_argoqc'
}

bbpFiltered = avh.query('argo', options=bbpQCfilteredQuery, apikey=API_KEY, apiroot=API_ROOT, verbose=True)
bbpindex = bbpFiltered[0]['data_info'][0].index('bbp700')
bbpQCindex = bbpFiltered[0]['data_info'][0].index('bbp700_argoqc')
print(bbpFiltered[0]['data'][bbpindex][0:10])
print(bbpFiltered[0]['data'][bbpQCindex][0:10])

https://argovis-api.colorado.edu/argo?id=2902857_001&data=bbp700,1,bbp700_argoqc
[0.004157, 0.005183, None, 0.005632, 0.005568, None, 0.004298, 0.004439, 0.004568, 0.004593]
[1, 1, 4, 1, 1, 4, 1, 1, 1, 1]


In our `data` query parameter, we listed which QC flags we find tolerable for each measurement parameter; in this case `bbp700,1` indicates we only want `bbp700` data if it has a corresponding QC flag of 1, and we only received those levels in our downloaded data. Some things implied by this example that are worth highlighting:

 - QC flags listed after a variable name only apply to that variable name. Try printing the `pressure` record for the profile found above, and you'll see none of its levels were suppressed.
 - The list of QC flags is an explicit-allow list and can contain as many flags as you want. For example, you might change the above data query to `bbp700,1,2` to get both 1- and 2-flagged `bbp700` measurements back.
 - We include the explicit QC flag in this example for illustrative purposes, but it's not required when doing QC filtering in this way. Try the above query while omitting `bbp700_argoqc`, and you'll get the same non-`None` values for `bbp700`.
 - Note however, as with all data requests, if all explicitly requested data variables are `None` for a level, that level is dropped. In the case where you omitted `bbp700_argoqc` and only requested `bbp700`, the levels where the QC filtration set the `bbp700` value to `None` are dropped.
 - Similarly, if *all* levels of a requested variable are set to `None` by QC filtration, the entire profile will be dropped from the returns, on the grounds that it doesn't contain any of the data you requested at a level of quality you marked as acceptable.
 
Note that QC flags are currently only available for the argo and cchdo datasets, and furthermore that these datasets assign different meanings to their QC flags. Be sure to check the docs for each project to make sure you understand how to interpret that project's QC flags.

### Minimal data responses

Sometimes, we might want to use the `data` filter as we've seen to confine our attention to only profiles that have data of interest, but we're only interested in general or metadata about those measurements, and don't want to download the actual measurements; for this, we can add the `except-data-values` token:

In [86]:
dataQuery = {
    'startDate': '2017-08-01T00:00:00Z',
    'endDate': '2017-09-01T00:00:00Z',
    'polygon': [[-150,-30],[-155,-30],[-155,-35],[-150,-35],[-150,-30]],
    'data': 'doxy,except-data-values'
}

profiles = avh.query('argo', options=dataQuery, apikey=API_KEY, apiroot=API_ROOT, verbose=True)
print(profiles)

https://argovis-api.colorado.edu/argo?startDate=2017-08-01T00:00:00Z&endDate=2017-09-01T00:00:00Z&polygon=[[-150,+-30],+[-155,+-30],+[-155,+-35],+[-150,+-35],+[-150,+-30]]&data=doxy,except-data-values
[{'_id': '5905107_001', 'geolocation': {'type': 'Point', 'coordinates': [-154.974, -32.415]}, 'basin': 2, 'timestamp': '2017-08-11T11:57:19.001Z', 'date_updated_argovis': '2026-04-10T21:47:19.238Z', 'source': [{'source': ['argo_core'], 'url': 'ftp://ftp.ifremer.fr/ifremer/argo/dac/aoml/5905107/profiles/D5905107_001.nc', 'date_updated': '2019-06-24T15:29:23.000Z'}, {'source': ['argo_bgc'], 'url': 'ftp://ftp.ifremer.fr/ifremer/argo/dac/aoml/5905107/profiles/SD5905107_001.nc', 'date_updated': '2025-12-26T15:26:16.000Z'}], 'cycle_number': 1, 'geolocation_argoqc': 1, 'profile_direction': 'A', 'timestamp_argoqc': 1, 'vertical_sampling_scheme': 'Primary sampling: mixed [deeper than nominal 985dbar: discrete; nominal 985dbar to surface: 2dbar-bin averaged]', 'data_info': [['doxy', 'pressure'], ['

Note that specifying only `'data': 'except-data-values'` is the same as just leaving the `data` query key off completely; the purpose of this option is to allow you to filter by data, but then only get back the lightweight non-measurement values. 

If we want an even more minimal response, we can use the `compression=minimal` option:

In [87]:
dataQuery = {
    'startDate': '2017-08-01T00:00:00Z',
    'endDate': '2017-09-01T00:00:00Z',
    'polygon': [[-150,-30],[-155,-30],[-155,-35],[-150,-35],[-150,-30]],
    'data': 'doxy',
    'compression': 'minimal'
}

profiles = avh.query('argo', options=dataQuery, apikey=API_KEY, apiroot=API_ROOT, verbose=True)
print(profiles)

https://argovis-api.colorado.edu/argo?startDate=2017-08-01T00:00:00Z&endDate=2017-09-01T00:00:00Z&polygon=[[-150,+-30],+[-155,+-30],+[-155,+-35],+[-150,+-35],+[-150,+-30]]&data=doxy&compression=minimal
[['5905107_001', -154.974, -32.415, '2017-08-11T11:57:19.001Z', ['argo_core', 'argo_bgc'], ['5905107_m0']]]


With `compression: minimal`, for each data document we get only a minimal amount of information describing it; each data product has a slightly different minimal representation tailored to suit.

### Temporospatial request details

You have seen in examples above that requests can be temporally limited by `startDate` and `endDate`, and confined to a geographic region with `polygon`. There are a few more features and facts about temporospatial requests in Argovis that are worth exploring.

#### Box regions

The `polygon` region definitions you've seen so far define regions on the globe by connecting vertexes with geodesic edges. If instead we want a region bounded by lines of constant latitude and longitude, there is the `box` query string parameter. Compare two similar but different searches, first with `polygon`, similar to the above, tracing geodesics between four corners of a region:

In [88]:
qs = {
    'startDate': '2017-08-01T00:00:00Z',
    'endDate': '2017-09-01T00:00:00Z',
    'polygon': [[-20,70],[20,70],[20,72],[-20,72],[-20,70]],
}

profiles = avh.query('argo', options=qs, apikey=API_KEY, apiroot=API_ROOT, verbose=True)
latitudes = [x['geolocation']['coordinates'][1] for x in profiles]
print(min(latitudes))
print(max(latitudes))

https://argovis-api.colorado.edu/argo?startDate=2017-08-01T00:00:00Z&endDate=2017-09-01T00:00:00Z&polygon=[[-20,+70],+[20,+70],+[20,+72],+[-20,+72],+[-20,+70]]
70.78334480553657
72.868775


Now try something similar, but with a `box` region defined instead across the four corners:

In [89]:
qs = {
    'startDate': '2017-08-01T00:00:00Z',
    'endDate': '2017-09-01T00:00:00Z',
    'box': [[-20,70],[20,72]]
}

profiles = avh.query('argo', options=qs, apikey=API_KEY, apiroot=API_ROOT, verbose=True)
latitudes = [x['geolocation']['coordinates'][1] for x in profiles]
print(min(latitudes))
print(max(latitudes))

https://argovis-api.colorado.edu/argo?startDate=2017-08-01T00:00:00Z&endDate=2017-09-01T00:00:00Z&box=[[-20,+70],+[20,+72]]
70.017
71.957


Notice that while both regions share the same corners, the polygon search actually returns profiles with latitudes higher than the region's northermost corners since geodesics between two points sharing a latitude deflect north in this far-north search region. Meanwhile, the latitudes of profiles in the box region are confined between the lines of constant latitude connecting the vertexes and defining the top and bottom of the box.

> **Box mode notation**: note that box mode expects exactly two vertexes: the most southern and western corner first, followed by the most northern and eastern corner.

### Metadata requests

We saw above using the `/<dataset>/meta` route to ask for specific metadata documents; however, it's useful to point out one piece of sugar to help download all the metadata documents that correspond to the data documents returned by a given query. Add the query string parameter `batchmeta=true` to your data query, and you will instead receive all the corresponding metadata documents:

In [90]:
qs = {
    'startDate': '2017-08-01T00:00:00Z',
    'endDate': '2017-09-01T00:00:00Z',
    'box': [[-20,70],[20,72]],
    'batchmeta': True
}

profile_metadata = avh.query('argo', options=qs, apikey=API_KEY, apiroot=API_ROOT, verbose=True)
print(profile_metadata[0])

https://argovis-api.colorado.edu/argo?startDate=2017-08-01T00:00:00Z&endDate=2017-09-01T00:00:00Z&box=[[-20,+70],+[20,+72]]&batchmeta=True
{'_id': '6902730_m0', 'data_type': 'oceanicProfile', 'data_center': 'IF', 'instrument': 'profiling_float', 'pi_name': ['Camille DAUBORD'], 'platform': '6902730', 'platform_type': 'ARVOR', 'fleetmonitoring': 'https://fleetmonitoring.euro-argo.eu/float/6902730', 'oceanops': 'https://www.ocean-ops.org/board/wa/Platform?ref=6902730', 'positioning_system': 'GPS', 'wmo_inst_type': '844'}


# Other schema

All Argovis datasets follow the patterns and query string logic illustrated above. Some datasets, however, have some subtle differences in their schema worth being aware of when working with the JSON directly.

## Sparse grids

Gridded products that cover a limited (O(10^6)) number of lat-long-timestamp points are essentially identical to the per-profile examples above, but importantly represent their standardized level spectrum on their metadata documents, so you aren't re-downloading it for every single profile. For example, the Roemmich-Gilson temperature grid:

In [91]:
qs = {
    'startDate': '2017-08-01T00:00:00Z',
    'endDate': '2017-09-01T00:00:00Z',
    'box': [[-20,-20],[-10,-10]],
    'data': 'all'
}

gridpoints = avh.query('grids/rg09', options=qs, apikey=API_KEY, apiroot=API_ROOT, verbose=True)

https://argovis-api.colorado.edu/grids/rg09?startDate=2017-08-01T00:00:00Z&endDate=2017-09-01T00:00:00Z&box=[[-20,+-20],+[-10,+-10]]&data=all


In [92]:
gridpoints[0]

{'_id': '20170815000000_-19.5_-18.5',
 'metadata': ['rg09_temperature_200401_Total', 'rg09_salinity_200401_Total'],
 'geolocation': {'type': 'Point', 'coordinates': [-19.5, -18.5]},
 'basin': 1,
 'timestamp': '2017-08-15T00:00:00.000Z',
 'data': [[22.541,
   22.558001,
   22.521,
   22.465,
   22.496,
   22.514,
   22.483,
   22.542999,
   22.506001,
   22.505999,
   22.57,
   22.539,
   22.323,
   21.915001,
   21.328001,
   20.429001,
   19.639999,
   18.772001,
   17.852999,
   16.787001,
   15.735,
   14.824,
   14.131001,
   13.525001,
   12.943,
   12.381001,
   11.818,
   11.25,
   10.736,
   10.217,
   9.707,
   9.212999,
   8.685,
   7.888,
   6.925,
   6.104,
   5.436,
   4.959,
   4.583,
   4.292,
   4.078,
   3.932,
   3.837,
   3.785,
   3.777,
   3.79,
   3.82,
   3.856,
   3.892,
   3.914,
   3.912,
   3.877,
   3.783,
   3.66,
   3.559,
   3.449,
   3.337,
   3.251],
  [36.764,
   36.762001,
   36.783001,
   36.787998,
   36.797001,
   36.794998,
   36.786003,
   36.791

In [93]:
qs = {
    'startDate': '2017-08-01T00:00:00Z',
    'endDate': '2017-09-01T00:00:00Z',
    'box': [[-20,-20],[-10,-10]],
    'data': 'all',
    'batchmeta': True
}

gridpoints_meta = avh.query('grids/rg09', options=qs, apikey=API_KEY, apiroot=API_ROOT, verbose=True)
print(gridpoints_meta[0]['levels'])

https://argovis-api.colorado.edu/grids/rg09?startDate=2017-08-01T00:00:00Z&endDate=2017-09-01T00:00:00Z&box=[[-20,+-20],+[-10,+-10]]&data=all&batchmeta=True
[2.5, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120, 130, 140, 150, 160, 170, 182.5, 200, 220, 240, 260, 280, 300, 320, 340, 360, 380, 400, 420, 440, 462.5, 500, 550, 600, 650, 700, 750, 800, 850, 900, 950, 1000, 1050, 1100, 1150, 1200, 1250, 1300, 1350, 1412.5, 1500, 1600, 1700, 1800, 1900, 1975]


This level spectrum works like a `pressure` vector in a profile's data attribute: the levels listed correspond 1:1 with the vectors in the associated data document, indicating the pressure of each measured level. Note that if you use a depth filter, the surviving levels will be copied to the data document, which then take precedence over the metadata:

In [94]:
qs = {
    'startDate': '2017-08-01T00:00:00Z',
    'endDate': '2017-09-01T00:00:00Z',
    'box': [[-20,-20],[-10,-10]],
    'data': 'all',
    'verticalRange':[0,1000]
}

gridpoints = avh.query('grids/rg09', options=qs, apikey=API_KEY, apiroot=API_ROOT, verbose=True)
gridpoints[0]

https://argovis-api.colorado.edu/grids/rg09?startDate=2017-08-01T00:00:00Z&endDate=2017-09-01T00:00:00Z&box=[[-20,+-20],+[-10,+-10]]&data=all&verticalRange=0&verticalRange=1000


{'_id': '20170815000000_-19.5_-18.5',
 'metadata': ['rg09_temperature_200401_Total', 'rg09_salinity_200401_Total'],
 'geolocation': {'type': 'Point', 'coordinates': [-19.5, -18.5]},
 'basin': 1,
 'timestamp': '2017-08-15T00:00:00.000Z',
 'data': [[22.541,
   22.558001,
   22.521,
   22.465,
   22.496,
   22.514,
   22.483,
   22.542999,
   22.506001,
   22.505999,
   22.57,
   22.539,
   22.323,
   21.915001,
   21.328001,
   20.429001,
   19.639999,
   18.772001,
   17.852999,
   16.787001,
   15.735,
   14.824,
   14.131001,
   13.525001,
   12.943,
   12.381001,
   11.818,
   11.25,
   10.736,
   10.217,
   9.707,
   9.212999,
   8.685,
   7.888,
   6.925,
   6.104,
   5.436,
   4.959,
   4.583,
   4.292,
   4.078,
   3.932,
   3.837],
  [36.764,
   36.762001,
   36.783001,
   36.787998,
   36.797001,
   36.794998,
   36.786003,
   36.791,
   36.793999,
   36.803997,
   36.813999,
   36.806999,
   36.764999,
   36.683998,
   36.565998,
   36.381001,
   36.224003,
   36.056999,
   35

## Timeseries

For high resolution grids, the number of lat-long-timestamp pairs can become so large as to be infeasible to index on our resources. We collapse the time dimension of these grids by representing them slightly differently, as *timeseries documents*, where the vectors in the data documents' data attribute correspond to timesteps. For example, the NOAA SST product:

In [95]:
qs = {
    'box': [[-20,-20],[-10,-10]],
    'data': 'all'
}

tspoints = avh.query('timeseries/noaasst', options=qs, apikey=API_KEY, apiroot=API_ROOT, verbose=True)

https://argovis-api.colorado.edu/timeseries/noaasst?box=[[-20,+-20],+[-10,+-10]]&data=all


In [96]:
tspoints[0]

{'_id': '-19.5_-19.5',
 'metadata': ['noaasst'],
 'basin': 1,
 'geolocation': {'type': 'Point', 'coordinates': [-19.5, -19.5]},
 'data': [[23.98,
   24.8,
   25.09,
   25.080000000000002,
   25.43,
   26.23,
   26.82,
   26.8,
   26.97,
   26.240000000000002,
   26.68,
   26.68,
   26.650000000000002,
   27.12,
   26.86,
   26.82,
   26.330000000000002,
   25.6,
   25.53,
   25.6,
   25.09,
   25.19,
   24.73,
   24.66,
   24.34,
   24.37,
   23.61,
   23.06,
   23.23,
   23.240000000000002,
   22.580000000000002,
   22.32,
   22.41,
   22.09,
   21.990000000000002,
   22.18,
   22.12,
   21.990000000000002,
   22.2,
   21.97,
   21.91,
   22.37,
   22.63,
   23.17,
   23.900000000000002,
   24.11,
   24.060000000000002,
   24.240000000000002,
   23.96,
   24.650000000000002,
   24.96,
   24.330000000000002,
   24.400000000000002,
   24.98,
   25.150000000000002,
   24.900000000000002,
   25.38,
   25.29,
   25.54,
   26.04,
   26.21,
   26.29,
   26.41,
   25.650000000000002,
   26.17

The data document contains a vector of SST measurements at the indicated geolocation; the timestamps of each of these measurements is given by the timeseries property of the corresponding metadata:

In [97]:
qs = {
    'box': [[-20,-20],[-10,-10]],
    'data': 'all',
    'batchmeta':'true'
}

tspoints_meta = avh.query('timeseries/noaasst', options=qs, apikey=API_KEY, apiroot=API_ROOT, verbose=True)
tspoints_meta

https://argovis-api.colorado.edu/timeseries/noaasst?box=[[-20,+-20],+[-10,+-10]]&data=all&batchmeta=true


[{'_id': 'noaasst',
  'data_type': 'noaa-oi-sst-v2',
  'data_info': [['sst'],
   ['units', 'long_name'],
   [['degC', 'Weekly Mean of Sea Surface Temperature']]],
  'date_updated_argovis': '2023-08-10T00:40:59.000Z',
  'timeseries': ['1989-12-31T00:00:00.000Z',
   '1990-01-07T00:00:00.000Z',
   '1990-01-14T00:00:00.000Z',
   '1990-01-21T00:00:00.000Z',
   '1990-01-28T00:00:00.000Z',
   '1990-02-04T00:00:00.000Z',
   '1990-02-11T00:00:00.000Z',
   '1990-02-18T00:00:00.000Z',
   '1990-02-25T00:00:00.000Z',
   '1990-03-04T00:00:00.000Z',
   '1990-03-11T00:00:00.000Z',
   '1990-03-18T00:00:00.000Z',
   '1990-03-25T00:00:00.000Z',
   '1990-04-01T00:00:00.000Z',
   '1990-04-08T00:00:00.000Z',
   '1990-04-15T00:00:00.000Z',
   '1990-04-22T00:00:00.000Z',
   '1990-04-29T00:00:00.000Z',
   '1990-05-06T00:00:00.000Z',
   '1990-05-13T00:00:00.000Z',
   '1990-05-20T00:00:00.000Z',
   '1990-05-27T00:00:00.000Z',
   '1990-06-03T00:00:00.000Z',
   '1990-06-10T00:00:00.000Z',
   '1990-06-17T00:00:00.0

Similar to filtering grids by level, filtering a timeseries by time range will subset and copy this timeseries vector to the resulting data documents, which then takes precedence for labeling the associated data vectors:

In [98]:
qs = {
    'box': [[-20,-20],[-10,-10]],
    'data': 'all',
    'startDate': '2020-01-01T00:00:00Z',
    'endDate': '2020-02-01T00:00:00Z'
}

tspoints = avh.query('timeseries/noaasst', options=qs, apikey=API_KEY, apiroot=API_ROOT, verbose=True)
tspoints[0]

https://argovis-api.colorado.edu/timeseries/noaasst?box=[[-20,+-20],+[-10,+-10]]&data=all&startDate=2020-01-01T00:00:00Z&endDate=2020-02-01T00:00:00Z


{'_id': '-19.5_-19.5',
 'metadata': ['noaasst'],
 'basin': 1,
 'geolocation': {'type': 'Point', 'coordinates': [-19.5, -19.5]},
 'data': [[25.6, 26.89, 26.84, 26.87]],
 'data_info': [['sst'],
  ['units', 'long_name'],
  [['degC', 'Weekly Mean of Sea Surface Temperature']]],
 'timeseries': ['2020-01-05T00:00:00.000Z',
  '2020-01-12T00:00:00.000Z',
  '2020-01-19T00:00:00.000Z',
  '2020-01-26T00:00:00.000Z']}